# Universidad del Gran Rosario - UGR

## Ingeniería del Software

### Trabajo Práctico Integrador

**Introducción a las metodologías y procesos de Ciencia de Datos**

**Docentes:** Ing. Ignacio Sanseovich - Lic. Briant Gauna

**Estudiantes:**

- Ivan Porcari
- Marcelo Saucedo
- Rodrigo Zamora
- Gaspar Giannitrapani
- Stefania Martos
- Juan Ignacio Sasia
- Juan José Muñoz Franchi
- Jorge Nicolas Segovia


# Estudio de caso

Este trabajo se basa en la aplicación de una metodología de minería de datos y su implementación en una solución de software.

El caso elegido se centra en el análisis de la demanda energética, tomando como punto de partida datos históricos de consumo. La idea principal es estudiar el comportamiento de la demanda a lo largo del tiempo, detectar patrones relevantes y evaluar si es posible construir un modelo que ayude a estimar períodos de mayor consumo.

Para ordenar el desarrollo del trabajo se utilizará la metodología CRISP-DM, ya que permite recorrer el proyecto desde la comprensión del problema hasta la evaluación de resultados y una posible implantación conceptual de la solución.


##Fases del Trabajo:


# Índice general

0. Configuración inicial
1. Comprensión del negocio
2. Comprensión de los datos
3. Preparación de los datos
4. Exploración de los datos
5. Modelado
6. Evaluación
7. Implantación conceptual
8. Conclusiones finales
9. Referencias y anexos


# 0. Configuración inicial

En esta sección se dejará preparado el entorno de trabajo del notebook.

Acá se van a importar las librerías necesarias, conectar Google Drive y definir las rutas principales del proyecto para trabajar siempre sobre la misma estructura de carpetas.

La idea es mantener separados los datos originales, los datos procesados y los resultados generados durante el análisis.


## 0.1 Importación de librerías

En esta parte importamos las librerías básicas que vamos a usar para trabajar con archivos, rutas y datos tabulares.


In [ ]:
from google.colab import files, userdata
import pandas as pd
from pathlib import Path

## 0.2 Verificación de acceso a la carpeta del proyecto

Como el proyecto está en una carpeta compartida de Google Drive, primero verificamos que Colab pueda acceder correctamente a la carpeta raíz del proyecto.

_Para que esta ruta funcione, la carpeta compartida **TPI** debe estar agregada como acceso directo dentro de **Mi unidad**._


In [ ]:

!git clone https://github.com/ju4n1t0x/eficiencia_energetica.git

Cloning into 'eficiencia_energetica'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 24 (delta 5), reused 23 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 876.88 KiB | 9.04 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [3]:
BASE_DIR = Path('/content/eficiencia_energetica')

BASE_DIR.exists()

True

## 0.3 Definición de rutas del proyecto

Una vez verificado el acceso a la carpeta principal, definimos las rutas que vamos a usar durante el trabajo.

Esto nos permite leer el dataset original, guardar datos procesados y exportar resultados sin escribir rutas manualmente en cada parte del notebook.


In [4]:
# Carpetas principales
DOCS_DIR = BASE_DIR / '00_documentacion'
NOTEBOOKS_DIR = BASE_DIR / '01_notebooks'
DATA_DIR = BASE_DIR / '02_data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'

OUTPUTS_DIR = BASE_DIR / '03_outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
TABLES_DIR = OUTPUTS_DIR / 'tables'
MODELS_DIR = OUTPUTS_DIR / 'models'

REPORTS_DIR = BASE_DIR / '04_reports'

## 0.4 Verificación de estructura de carpetas

Verificamos que las carpetas principales existan. En caso de que alguna carpeta de resultados todavía no esté creada, se crea automáticamente.


In [5]:
folders = [
    DOCS_DIR,
    NOTEBOOKS_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    MODELS_DIR,
    REPORTS_DIR
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(folder, "OK" if folder.exists() else "NO EXISTE")

/content/eficiencia_energetica/00_documentacion OK
/content/eficiencia_energetica/01_notebooks OK
/content/eficiencia_energetica/02_data/raw OK
/content/eficiencia_energetica/02_data/processed OK
/content/eficiencia_energetica/03_outputs/figures OK
/content/eficiencia_energetica/03_outputs/tables OK
/content/eficiencia_energetica/03_outputs/models OK
/content/eficiencia_energetica/04_reports OK


---


# 1. Comprensión del negocio

En esta primera fase se busca entender el problema desde una mirada general, antes de pasar al análisis de datos.

El objetivo es explicar qué problema se quiere abordar, por qué tiene sentido trabajarlo, quiénes podrían beneficiarse con una solución de este tipo y qué se espera lograr con el proyecto.


## 1.1 Contexto del problema

El consumo energético es una variable importante tanto para los usuarios como para las empresas distribuidoras y los organismos encargados de la planificación energética.

Cuando la demanda aumenta en ciertos períodos, puede ser necesario anticipar recursos, reforzar infraestructura o tomar decisiones operativas con mayor información.

Por este motivo, contar con un análisis de la demanda histórica puede ayudar a entender mejor el comportamiento del consumo y detectar momentos de mayor exigencia para el sistema.


## 1.2 Problema identificado

El problema que se busca abordar es la dificultad para anticipar períodos de mayor demanda energética a partir de datos históricos.

Si bien existen registros de consumo, estos datos por sí solos no siempre son fáciles de interpretar. Por eso se propone aplicar un proceso de análisis y minería de datos que permita encontrar patrones, comparar comportamientos y generar una base para posibles predicciones.


## 1.3 Stakeholders

Los principales actores relacionados con este problema son:

- Consumidores finales.
- Distribuidoras eléctricas.
- Gobierno u organismos reguladores.
- Equipos técnicos y operativos.
- Equipo desarrollador del proyecto.

Cada uno de estos actores puede beneficiarse de distintas formas. Los consumidores pueden verse favorecidos por una mejor planificación del servicio, las distribuidoras pueden anticipar períodos críticos y los organismos públicos pueden contar con información útil para la toma de decisiones.


## 1.4 Objetivo general

Analizar datos históricos de demanda energética para identificar patrones de consumo y evaluar la posibilidad de construir un modelo que permita estimar períodos de mayor demanda.


## 1.5 Objetivos específicos

- Analizar la evolución de la demanda energética a lo largo del tiempo.
- Identificar variaciones de consumo por año, mes o estación.
- Comparar el comportamiento de la demanda según las categorías disponibles en el dataset.
- Detectar posibles patrones o tendencias relevantes.
- Construir un modelo base que permita estimar la demanda energética.
- Evaluar los resultados obtenidos y sus limitaciones.


## 1.6 Alcance del proyecto

El alcance principal del trabajo será el análisis y modelado de la demanda energética a partir del dataset seleccionado.

El proyecto no busca predecir cortes eléctricos de forma directa, ya que para eso serían necesarios datos específicos sobre interrupciones del servicio. En cambio, se trabajará sobre la demanda energética como variable principal, entendiendo que su análisis puede servir como apoyo para la planificación y prevención de escenarios de alta exigencia.


## 1.7 Criterios de éxito del negocio

Desde el punto de vista del negocio, el proyecto será exitoso si permite:

- Comprender mejor el comportamiento histórico de la demanda energética.
- Identificar períodos de mayor consumo.
- Generar información útil para apoyar la toma de decisiones.
- Presentar los resultados de forma clara para usuarios técnicos y no técnicos.


---


# 2. Comprensión de los datos

En esta fase se realiza el primer acercamiento al dataset.

La idea es conocer de dónde provienen los datos, qué información contienen, qué columnas tienen, qué período cubren y qué problemas podrían aparecer antes de avanzar con la limpieza y el modelado.


## 2.1 Fuente de datos

El dataset utilizado corresponde a registros históricos de demanda energética.

En esta sección se documentará la fuente del archivo, el nombre del dataset, el período cubierto, la unidad de medida y las principales variables disponibles.


## 2.2 Carga inicial del dataset

En esta sección cargamos el archivo original desde la carpeta correspondiente del proyecto.

El dataset original se mantiene sin modificaciones dentro de la carpeta de datos crudos, y cualquier transformación posterior se guardará como una nueva versión procesada.


In [6]:
csv_files = list(RAW_DATA_DIR.glob('*.csv'))
csv_files

[PosixPath('/content/eficiencia_energetica/02_data/raw/demanda-ltimos-aos.csv')]

Seleccionamos el archivo encontrado para cargarlo en el notebook.


In [7]:
csv_path = csv_files[0]
csv_path

PosixPath('/content/eficiencia_energetica/02_data/raw/demanda-ltimos-aos.csv')

Cargamos el dataset en un DataFrame y mostramos las primeras filas para verificar la lectura.


In [8]:
df = pd.read_csv(csv_path)
df.head()

,id,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_MWh,fecha_proceso,lote_id_log,indice_tiempo
0,699232,2017,1,AARGTAOY,AEROP ARG 2000 - Aeroparque,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1990.439,2020-05-05 11:06:49.000713,67,2017-01
1,699233,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1609.464,2020-05-05 11:06:49.000713,67,2017-01
2,699234,2017,1,ACARQQ3Y,ASOC.COOP.ARG. - Quequén,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,421.334,2020-05-05 11:06:49.000713,67,2017-01
3,699235,2017,1,ACARSLSY,ASOC.COOP.ARG. - San Lorenzo,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,449.145,2020-05-05 11:06:49.000713,67,2017-01
4,699236,2017,1,ACERBR1Y,Planta Bragado,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,18073.170,2020-05-05 11:06:49.000713,67,2017-01


## 2.3 Descripción inicial de los datos

En esta parte se revisarán las primeras filas del dataset, la cantidad de registros, la cantidad de columnas y los tipos de datos disponibles.

Esto permitirá tener una primera idea de la estructura general del archivo antes de aplicar limpieza o transformaciones.


## 2.4 Diccionario de datos

Se armará un diccionario simple con las columnas principales del dataset.

El objetivo es explicar qué significa cada variable, qué tipo de dato contiene y si será utilizada para el análisis exploratorio o para el modelo.


## 2.5 Verificación inicial de calidad

En esta sección se revisará si existen valores nulos, duplicados, fechas inconsistentes, valores negativos o categorías mal registradas.

Esta revisión es importante para decidir qué transformaciones serán necesarias en la fase de preparación de datos.


---


# 3. Preparación de los datos

En esta fase se preparará el dataset para que pueda ser utilizado en el análisis exploratorio y en el modelado.

La preparación puede incluir selección de columnas, limpieza de valores, corrección de formatos, creación de nuevas variables y guardado de una versión limpia del dataset.


## 3.1 Selección de datos

Se seleccionarán las columnas necesarias para responder los objetivos del proyecto.

También se identificarán columnas que no aporten al análisis o que no puedan utilizarse por problemas de calidad, formato o falta de relación con el objetivo.


## 3.2 Limpieza de datos

Se aplicarán las correcciones necesarias sobre el dataset.

Esto puede incluir tratamiento de nulos, eliminación de duplicados, corrección de nombres de columnas, revisión de categorías y control de valores fuera de rango.


## 3.3 Construcción de nuevas variables

A partir de las columnas existentes se podrán crear nuevas variables útiles para el análisis.

Por ejemplo, si el dataset contiene fechas, se podrán crear columnas de año, mes o estación para estudiar el comportamiento temporal de la demanda.


## 3.4 Formateo final del dataset

Se revisará que el dataset quede en un formato adecuado para los gráficos y el modelo.

La versión final procesada se guardará en la carpeta de datos limpios para reutilizarla en las siguientes etapas del trabajo.


---


# 4. Exploración de los datos

En esta fase se analizarán los datos ya preparados para encontrar patrones, tendencias o comportamientos relevantes.

El análisis exploratorio permitirá validar o ajustar las hipótesis iniciales y entender mejor la demanda energética antes de construir el modelo.


## 4.1 Consumo energético por año

Se analizará la evolución de la demanda energética a lo largo de los años disponibles en el dataset.


## 4.2 Consumo energético por mes

Se revisará si existen meses con mayor o menor consumo, buscando detectar posibles comportamientos estacionales.


## 4.3 Consumo por categoría

Se comparará el consumo según las categorías disponibles en el dataset, como residencial, comercial o industrial, en caso de que estén presentes.


## 4.4 Validación de hipótesis

Se revisarán las hipótesis planteadas al inicio del trabajo y se evaluará si los datos permiten confirmarlas, rechazarlas o ajustarlas parcialmente.


---


# 5. Modelado

En esta fase se construirá un modelo base para estimar la demanda energética.

La idea no es buscar el modelo más complejo, sino construir una primera solución clara, explicable y alineada con el objetivo del proyecto.


## 5.1 Selección de la variable objetivo

Se definirá cuál será la variable que el modelo intentará estimar.

En este proyecto, la variable objetivo estará relacionada con la demanda o consumo energético.


## 5.2 Selección de variables predictoras

Se elegirán las variables que podrían ayudar al modelo a estimar la demanda.

Estas variables pueden estar relacionadas con el tiempo, la categoría tarifaria u otras columnas disponibles en el dataset.


## 5.3 División de datos de entrenamiento y prueba

Se dividirán los datos en un conjunto de entrenamiento y otro de prueba.

Esto permitirá entrenar el modelo con una parte de los datos y evaluar su comportamiento con datos que no vio durante el entrenamiento.


## 5.4 Entrenamiento del modelo base

Se entrenará un primer modelo simple para generar predicciones iniciales de demanda energética.


---


# 6. Evaluación

En esta fase se analizará qué tan bien funciona el modelo construido.

No alcanza con generar predicciones: también es necesario medir el error, comparar valores reales contra valores estimados e interpretar si el resultado es útil para el objetivo planteado.


## 6.1 Métricas de evaluación

Se calcularán métricas que permitan medir el error del modelo, como MAE, RMSE y R2 si corresponde.


## 6.2 Comparación entre valores reales y predichos

Se compararán los resultados generados por el modelo contra los valores reales del dataset.


## 6.3 Interpretación de resultados

Se explicará en palabras simples si el modelo funciona de manera aceptable, dónde puede fallar y qué limitaciones tiene.


---


# 7. Implantación conceptual

En esta fase se explicará cómo podría utilizarse la solución en un escenario real.

No necesariamente se desarrollará una aplicación completa, pero sí se describirá cómo los resultados del análisis podrían convertirse en una herramienta útil para la toma de decisiones.


## 7.1 Uso esperado de la solución

La solución podría ser utilizada por una distribuidora eléctrica, un equipo técnico o un área de planificación para consultar tendencias de consumo y estimaciones de demanda.


## 7.2 Entrada y salida del sistema

Como entrada, el sistema utilizaría datos históricos de demanda energética.

Como salida, podría mostrar gráficos, indicadores, períodos de mayor consumo y estimaciones generadas por el modelo.


## 7.3 Posibles mejoras futuras

Como mejora futura se podría sumar una interfaz simple, integrar datos climáticos, permitir la carga de nuevos archivos o desplegar una versión web del análisis.


---


# 8. Conclusiones finales

En esta sección se resumirán los principales resultados del trabajo.

Se retomará el problema inicial, se explicará qué se pudo observar en los datos, qué aportó el modelo y qué limitaciones quedaron para futuras mejoras.


## 8.1 Principales hallazgos

Se resumirán los patrones o comportamientos más importantes encontrados durante el análisis.


## 8.2 Limitaciones

Se mencionarán las limitaciones del dataset, del modelo y del alcance general del trabajo.


## 8.3 Próximos pasos

Se dejarán planteadas posibles mejoras para una segunda etapa del proyecto.


---


# 9. Referencias y anexos

En esta sección se incluirán las fuentes utilizadas para el dataset, material de referencia de la metodología CRISP-DM y cualquier anexo que ayude a entender el trabajo.
